In [53]:
# Simular 60 días de precios de una acción
import numpy as np

np.random.seed(42)  # Para reproducibilidad

# Generar fechas
fechas = pd.date_range(start='2024-01-01', periods=60, freq='B')  # B = días hábiles

# Generar precios con tendencia alcista y volatilidad
precio_inicial = 100
rendimientos_simulados = np.random.normal(0.002, 0.02, 60)  # Media 0.2%, std 2%
precios_simulados = precio_inicial * np.cumprod(1 + rendimientos_simulados)

# Crear Serie de precios
PRECIOS_ACCION = pd.Series(
    precios_simulados.round(2),
    index=fechas,
    name='ACME Corp'
)

print("Precios de ACME Corp (primeros 10 días):")
print(PRECIOS_ACCION.head(10))
print(f"\nTotal de días: {len(PRECIOS_ACCION)}")

Precios de ACME Corp (primeros 10 días):
2024-01-01    101.19
2024-01-02    101.12
2024-01-03    102.63
2024-01-04    105.96
2024-01-05    105.68
2024-01-08    105.39
2024-01-09    108.93
2024-01-10    110.82
2024-01-11    110.00
2024-01-12    111.42
Freq: B, Name: ACME Corp, dtype: float64

Total de días: 60


In [54]:
# Datos adicionales para pruebas más completas
np.random.seed(123)

# Acción con alta volatilidad
rend_volatil = np.random.normal(0, 0.05, 60)  # 5% de volatilidad diaria
precios_volatil = 50 * np.cumprod(1 + rend_volatil)
ACCION_VOLATIL = pd.Series(
    precios_volatil.round(2),
    index=fechas,
    name='VolatilTech'
)

# Acción con tendencia bajista
rend_bajista = np.random.normal(-0.005, 0.015, 60)  # Tendencia negativa
precios_bajista = 200 * np.cumprod(1 + rend_bajista)
ACCION_BAJISTA = pd.Series(
    precios_bajista.round(2),
    index=fechas,
    name='DeclineCorp'
)

print("Acciones disponibles para análisis:")
print(f"1. ACME Corp - Precio actual: ${PRECIOS_ACCION.iloc[-1]:.2f}")
print(f"2. VolatilTech - Precio actual: ${ACCION_VOLATIL.iloc[-1]:.2f}")
print(f"3. DeclineCorp - Precio actual: ${ACCION_BAJISTA.iloc[-1]:.2f}")

Acciones disponibles para análisis:
1. ACME Corp - Precio actual: $92.74
2. VolatilTech - Precio actual: $59.42
3. DeclineCorp - Precio actual: $138.97


In [55]:
import pandas as pd

def bandas_bollinger(precios, ventana=20, k=2):
    """
    Calcula las Bandas de Bollinger para una Serie de precios.
    - precios: pd.Series con los precios de cierre.
    - ventana: periodos para la media móvil (típicamente 20).
    - k: número de desviaciones estándar (típicamente 2).
    """
    # 1. Calculamos la media móvil (Banda Media)
    media_movil = precios.rolling(window=ventana).mean()
    
    # 2. Calculamos la desviación estándar móvil
    std_movil = precios.rolling(window=ventana).std()
    
    # 3. Calculamos las bandas superior e inferior
    banda_sup = media_movil + (k * std_movil)
    banda_inf = media_movil - (k * std_movil)
    
    # Retornamos un DataFrame con las tres series
    return pd.DataFrame({
        'Banda_Sup': banda_sup,
        'Media': media_movil,
        'Banda_Inf': banda_inf
    })

In [56]:
def clasificar_volatilidad(precios, ventana=20):
    # Calculamos el rendimiento porcentual y su desviación estándar
    volatilidad = precios.pct_change().rolling(window=ventana).std()
    
    # Usamos pd.cut para categorizar automáticamente
    # Definimos límites (esto es un ejemplo, ajusta según tus datos)
    bins = [0, 0.01, 0.02, np.inf]
    labels = ['Baja', 'Media', 'Alta']
    
    return pd.cut(volatilidad, bins=bins, labels=labels)

In [57]:
def estadisticas_basicas(precios: pd.Series) -> Dict:
    """
    Calcula estadísticas descriptivas de los precios usando métodos de Pandas.
    """
    resultado = {
        "precio_actual": precios.iloc[-1],
        "precio_minimo": precios.min(),
        "precio_maximo": precios.max(),
        "precio_promedio": precios.mean(),
        "precio_mediana": precios.median(),
        "desviacion_std": precios.std(),
        "rango": precios.max() - precios.min(),
        "dias_analizados": len(precios)
    }
    
    return resultado

# Para verificar que funciona, puedes ejecutar esto en la siguiente celda:
stats = estadisticas_basicas(PRECIOS_ACCION)
print("Resultados para ACME Corp:")
for clave, valor in stats.items():
    print(f"{clave}: {valor}")

Resultados para ACME Corp:
precio_actual: 92.74
precio_minimo: 86.67
precio_maximo: 111.42
precio_promedio: 96.88983333333334
precio_mediana: 95.645
desviacion_std: 7.128231646939469
rango: 24.75
dias_analizados: 60


In [58]:
def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.
    Fórmula: ((precio_hoy / precio_ayer) - 1) * 100
    """
    # .pct_change() hace exactamente el cálculo (precio_hoy - precio_ayer) / precio_ayer
    # Lo multiplicamos por 100 para tenerlo en formato de porcentaje.
    rendimientos = precios.pct_change() * 100
    
    # El primer elemento siempre será NaN porque no hay un "día anterior"
    # Es una buena práctica dejarlo así, pero si quisieras, podrías rellenarlo con 0
    return rendimientos

# Probemos la función con nuestra acción ACME Corp:
rendimientos_acme = calcular_rendimientos(PRECIOS_ACCION)
print("Primeros 5 rendimientos (%):")
print(rendimientos_acme.head())

Primeros 5 rendimientos (%):
2024-01-01         NaN
2024-01-02   -0.069177
2024-01-03    1.493275
2024-01-04    3.244665
2024-01-05   -0.264251
Freq: B, Name: ACME Corp, dtype: float64


In [59]:
def analisis_rendimientos(rendimientos: pd.Series) -> Dict:
    """Analiza los rendimientos calculados."""
    return {
        "rendimiento_total": rendimientos.sum(), # Suma de rendimientos diarios
        "rendimiento_promedio": rendimientos.mean(),
        "mejor_dia": (rendimientos.idxmax(), rendimientos.max()), # idxmax da la fecha
        "peor_dia": (rendimientos.idxmin(), rendimientos.min()),
        "dias_positivos": (rendimientos > 0).sum(),
        "dias_negativos": (rendimientos < 0).sum(),
        "volatilidad": rendimientos.std() # Desv. std de los rendimientos
    }

# Pruébalo:
resumen = analisis_rendimientos(rendimientos_acme)
print("\nResumen de rendimientos de ACME Corp:")
print(resumen)


Resumen de rendimientos de ACME Corp:
{'rendimiento_total': np.float64(-7.748229086684423), 'rendimiento_promedio': np.float64(-0.1313259167234648), 'mejor_dia': (Timestamp('2024-02-13 00:00:00'), np.float64(3.905831995719633)), 'peor_dia': (Timestamp('2024-02-21 00:00:00'), np.float64(-3.714554776603529)), 'dias_positivos': np.int64(26), 'dias_negativos': np.int64(33), 'volatilidad': np.float64(1.823543256771936)}


In [60]:
def alertas_precio(precios: pd.Series, umbral: float = 0.05) -> list:
    """
    Detecta si el precio ha variado más de un umbral respecto al inicio.
    """
    precio_actual = precios.iloc[-1]
    precio_inicio = precios.iloc[0]
    variacion = (precio_actual - precio_inicio) / precio_inicio
    
    alertas = []
    if variacion > umbral:
        alertas.append(f"Alerta: Subida significativa del {variacion*100:.1f}%")
    elif variacion < -umbral:
        alertas.append(f"Alerta: Bajada significativa del {abs(variacion)*100:.1f}%")
        
    return alertas

In [61]:
# PARTE 2: Indicadores Técnicos

def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """Calcula la media móvil simple (SMA)."""
    # .rolling().mean() crea la ventana deslizante y calcula el promedio
    return precios.rolling(window=ventana).mean()

def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> Dict:
    """Calcula las Bandas de Bollinger."""
    media = media_movil(precios, ventana)
    std = precios.rolling(window=ventana).std()
    
    return {
        "banda_superior": media + (num_std * std),
        "banda_media": media,
        "banda_inferior": media - (num_std * std)
    }

def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """Clasifica la tendencia actual basándose en la MA y el precio."""
    ma = media_movil(precios, ventana)
    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2]
    
    if precio_actual > ma_actual and ma_actual > ma_anterior:
        return "ALCISTA"
    elif precio_actual < ma_actual and ma_actual < ma_anterior:
        return "BAJISTA"
    else:
        return "LATERAL"

In [62]:
# PARTE 3: Sistema de Alertas

def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """Genera señales de COMPRA, VENTA o MANTENER."""
    ma_c = precios.rolling(window=ma_corta).mean()
    ma_l = precios.rolling(window=ma_larga).mean()
    
    # Creamos una serie vacía de "MANTENER"
    senales = pd.Series("MANTENER", index=precios.index)
    
    # Lógica de cruce:
    # COMPRA si la MA corta cruza hacia arriba a la larga
    # VENTA si la MA corta cruza hacia abajo a la larga
    cruce_arriba = (ma_c > ma_l) & (ma_c.shift(1) <= ma_l.shift(1))
    cruce_abajo = (ma_c < ma_l) & (ma_c.shift(1) >= ma_l.shift(1))
    
    senales[cruce_arriba] = "COMPRA"
    senales[cruce_abajo] = "VENTA"
    
    return senales

def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """Clasifica la volatilidad usando pd.cut."""
    std = rendimientos.std()
    # Usamos bins para clasificar según los criterios del reto
    bins = [0, 1, 3, 5, np.inf]
    labels = ["BAJA", "MEDIA", "ALTA", "MUY ALTA"]
    # pd.cut devuelve una categoría, la convertimos a string
    return pd.cut([std], bins=bins, labels=labels)[0]

In [63]:
def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> Dict:
    """
    Genera un reporte completo de análisis integrando todas las funciones anteriores.
    """
    # 1. Cálculos base necesarios
    rendimientos = calcular_rendimientos(precios)
    stats = estadisticas_basicas(precios)
    analisis_rend = analisis_rendimientos(rendimientos)
    
    # 2. Construcción del diccionario de reporte
    reporte = {
        "nombre": nombre_accion,
        "periodo": {
            "inicio": precios.index.min().strftime('%Y-%m-%d'),
            "fin": precios.index.max().strftime('%Y-%m-%d'),
            "dias": len(precios)
        },
        "estadisticas": stats,
        "rendimientos": analisis_rend,
        "tendencia": clasificar_tendencia(precios),
        "volatilidad": clasificar_volatilidad(rendimientos),
        "senal_actual": generar_senales_trading(precios).iloc[-1],
        "alertas_recientes": alertas_precio(precios)
    }
    
    return reporte

In [51]:
def mostrar_reporte(reporte):
    """Muestra el reporte de forma organizada."""
    print(f"╔════════════════════════════════════════════════════╗")
    print(f"║  REPORTE DE ACCIÓN: {reporte['nombre']:<25} ║")
    print(f"╠════════════════════════════════════════════════════╣")
    print(f"║ Periodo: {reporte['periodo']['inicio']} al {reporte['periodo']['fin']}")
    print(f"║ Tendencia actual: {reporte['tendencia']}")
    print(f"║ Volatilidad: {reporte['volatilidad']}")
    print(f"║ Señal de trading: {reporte['senal_actual']}")
    print(f"║----------------------------------------------------")
    print(f"║ Estadísticas:")
    for key, val in reporte['estadisticas'].items():
        if isinstance(val, float):
            print(f"║  - {key}: {val:.2f}")
        else:
            print(f"║  - {key}: {val}")
    print(f"╚════════════════════════════════════════════════════╝")

In [65]:
# Generar reporte para ACME
reporte_final = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
mostrar_reporte(reporte_final)

╔════════════════════════════════════════════════════╗
║  REPORTE DE ACCIÓN: ACME Corp                 ║
╠════════════════════════════════════════════════════╣
║ Periodo: 2024-01-01 al 2024-03-22
║ Tendencia actual: ALCISTA
║ Volatilidad: MEDIA
║ Señal de trading: MANTENER
║----------------------------------------------------
║ Estadísticas:
║  - precio_actual: 92.74
║  - precio_minimo: 86.67
║  - precio_maximo: 111.42
║  - precio_promedio: 96.89
║  - precio_mediana: 95.64
║  - desviacion_std: 7.13
║  - rango: 24.75
║  - dias_analizados: 60
╚════════════════════════════════════════════════════╝
